# Credit Default Risk Model
### Predicting borrower default probability using Logistic Regression, Decision Tree, and XGBoost

**Dataset:** Give Me Some Credit (Kaggle / public mirror)  
**Goal:** Build a production-style credit risk model that segments borrowers into Low, Medium, and High risk tiers based on predicted default probability.

**Tools:** Python, pandas, scikit-learn, XGBoost, matplotlib, seaborn


---
## 1. Setup and Data Loading


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

print("All libraries loaded successfully.")


### Loading the dataset

The Give Me Some Credit dataset contains 150,000 borrower records with financial attributes.  
Target variable: `SeriousDlqin2yrs` — 1 if the borrower defaulted within 2 years, 0 otherwise.

**Download link:** https://www.kaggle.com/c/GiveMeSomeCredit/data  
Save `cs-training.csv` in the same folder as this notebook and run the cell below.


In [ ]:
# Upload cs-training.csv when prompted
from google.colab import files
uploaded = files.upload()

# Load dataset
import pandas as pd
df = pd.read_csv('cs-training.csv', index_col=0)

print(f'Dataset shape: {df.shape}')
print(f'\nColumn names:')
print(df.columns.tolist())
print(f'\nFirst 5 rows:')
df.head()


In [ ]:
# Rename columns for clarity
df.rename(columns={
    'SeriousDlqin2yrs': 'default',
    'RevolvingUtilizationOfUnsecuredLines': 'revolving_utilization',
    'age': 'age',
    'NumberOfTime30-59DaysPastDueNotWorse': 'past_due_30_59',
    'DebtRatio': 'debt_ratio',
    'MonthlyIncome': 'monthly_income',
    'NumberOfOpenCreditLinesAndLoans': 'open_credit_lines',
    'NumberOfTimes90DaysLate': 'times_90_days_late',
    'NumberRealEstateLoansOrLines': 'real_estate_loans',
    'NumberOfTime60-89DaysPastDueNotWorse': 'past_due_60_89',
    'NumberOfDependents': 'dependents'
}, inplace=True)

print("Columns renamed.")
print(df.columns.tolist())


---
## 2. Exploratory Data Analysis (EDA)

Before building any model, we need to understand the data — its shape, distributions, missing values, and class imbalance.  
Skipping EDA and jumping straight to modelling is the most common mistake in data projects.


In [ ]:
# Basic stats
print("=== Dataset Info ===")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\n=== Default Rate ===")
default_rate = df['default'].mean()
print(f"Default rate: {default_rate:.2%}")
print(f"Non-default: {(1-default_rate):.2%}")
print(f"\n=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])


In [ ]:
# Visualise class imbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Default distribution
counts = df['default'].value_counts()
axes[0].bar(['No Default (0)', 'Default (1)'], counts.values,
            color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=0.5)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Age distribution by default
df.groupby('default')['age'].plot.hist(alpha=0.6, bins=40, ax=axes[1],
                                        legend=True, color=['#4C72B0', '#DD8452'])
axes[1].set_title('Age Distribution by Default Status')
axes[1].set_xlabel('Age')
axes[1].legend(['No Default', 'Default'])

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Class imbalance ratio: {counts[0]/counts[1]:.1f}:1 (non-default to default)")


In [ ]:
# Correlation heatmap
plt.figure(figsize=(11, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 9})
plt.title('Feature Correlation Matrix', fontsize=14, pad=12)
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Distribution of key financial features
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
features = ['revolving_utilization', 'debt_ratio', 'monthly_income',
            'past_due_30_59', 'times_90_days_late', 'open_credit_lines']

for ax, feature in zip(axes.flatten(), features):
    df[feature].clip(upper=df[feature].quantile(0.99)).hist(
        bins=50, ax=ax, color='#4C72B0', edgecolor='white', linewidth=0.3)
    ax.set_title(feature.replace('_', ' ').title())
    ax.set_xlabel('')

plt.suptitle('Distribution of Key Features (clipped at 99th percentile)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 3. Data Cleaning and Missing Value Treatment

Real credit data is messy. Missing income data is common because not all applicants disclose it.  
We handle this carefully — imputing with the median is standard practice for income fields in credit scoring.


In [ ]:
# Missing value treatment

# Monthly income: impute with median (standard for credit data)
median_income = df['monthly_income'].median()
df['monthly_income'].fillna(median_income, inplace=True)
print(f"Monthly income: imputed {df['monthly_income'].isnull().sum()} missing values with median = {median_income:,.0f}")

# Dependents: impute with 0 (most common value, reasonable default)
df['dependents'].fillna(0, inplace=True)
print(f"Dependents: imputed missing values with 0")

# Verify no missing values remain
print(f"\nRemaining missing values: {df.isnull().sum().sum()}")


In [ ]:
# Outlier treatment
# Cap extreme values at 99th percentile to prevent them from dominating the model

cols_to_cap = ['revolving_utilization', 'debt_ratio', 'monthly_income',
               'past_due_30_59', 'past_due_60_89', 'times_90_days_late']

for col in cols_to_cap:
    cap = df[col].quantile(0.99)
    original_outliers = (df[col] > cap).sum()
    df[col] = df[col].clip(upper=cap)
    print(f"{col}: capped {original_outliers} values above {cap:.2f}")

print("\nOutlier treatment complete.")


---
## 4. Feature Engineering

This is where good analysts separate themselves. Instead of just feeding raw features to the model,  
we create derived features that capture meaningful credit risk signals.


In [ ]:
# Feature engineering

# Debt-to-income ratio (classic credit risk metric)
# Monthly debt obligation estimated from debt_ratio * income
df['monthly_debt'] = df['debt_ratio'] * df['monthly_income']
df['dti_ratio'] = np.where(
    df['monthly_income'] > 0,
    df['monthly_debt'] / df['monthly_income'],
    0
)

# Total delinquency count (composite risk signal)
df['total_delinquencies'] = (
    df['past_due_30_59'] +
    df['past_due_60_89'] +
    df['times_90_days_late']
)

# Delinquency flag (binary: has the borrower ever been late?)
df['has_delinquency'] = (df['total_delinquencies'] > 0).astype(int)

# High utilization flag (>80% revolving credit used is a risk signal)
df['high_utilization'] = (df['revolving_utilization'] > 0.8).astype(int)

# Income buckets (risk varies significantly by income band)
df['income_bucket'] = pd.cut(
    df['monthly_income'],
    bins=[0, 3000, 6000, 10000, float('inf')],
    labels=['Low', 'Medium', 'High', 'Very High']
).astype(str)

# Age risk flag (very young borrowers have higher default rates historically)
df['young_borrower'] = (df['age'] < 25).astype(int)

print("Engineered features:")
new_features = ['monthly_debt', 'dti_ratio', 'total_delinquencies',
                'has_delinquency', 'high_utilization', 'income_bucket', 'young_borrower']
print(df[new_features].describe())


In [ ]:
# Visualise engineered features vs default rate
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Default rate by income bucket
income_default = df.groupby('income_bucket')['default'].mean().reindex(
    ['Low', 'Medium', 'High', 'Very High'])
axes[0].bar(income_default.index, income_default.values * 100,
            color=['#DD8452', '#4C72B0', '#55A868', '#C44E52'])
axes[0].set_title('Default Rate by Income Bucket')
axes[0].set_ylabel('Default Rate (%)')
axes[0].set_xlabel('Income Bucket')

# Default rate by delinquency flag
del_default = df.groupby('has_delinquency')['default'].mean()
axes[1].bar(['No Delinquency', 'Has Delinquency'], del_default.values * 100,
            color=['#4C72B0', '#DD8452'])
axes[1].set_title('Default Rate: Delinquency Flag')
axes[1].set_ylabel('Default Rate (%)')

# Default rate by high utilization
util_default = df.groupby('high_utilization')['default'].mean()
axes[2].bar(['Normal Utilization', 'High Utilization (>80%)'], util_default.values * 100,
            color=['#4C72B0', '#DD8452'])
axes[2].set_title('Default Rate: Revolving Utilization')
axes[2].set_ylabel('Default Rate (%)')

plt.tight_layout()
plt.savefig('feature_engineering_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 5. Model Building

We build three models in increasing complexity:
1. **Logistic Regression** -- baseline, interpretable, fast
2. **Decision Tree** -- captures non-linear relationships, easy to explain to stakeholders
3. **XGBoost** -- best performance, industry standard for credit scoring

We use `class_weight='balanced'` / `scale_pos_weight` to handle class imbalance.


In [ ]:
# Prepare features and target
feature_cols = [
    'revolving_utilization', 'age', 'past_due_30_59', 'debt_ratio',
    'monthly_income', 'open_credit_lines', 'times_90_days_late',
    'real_estate_loans', 'past_due_60_89', 'dependents',
    'dti_ratio', 'total_delinquencies', 'has_delinquency',
    'high_utilization', 'young_borrower', 'monthly_debt'
]

X = df[feature_cols]
y = df['default']

# Train/test split -- 80/20, stratified to preserve default rate in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")
print(f"Default rate in train: {y_train.mean():.2%}")
print(f"Default rate in test:  {y_test.mean():.2%}")


In [ ]:
# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Calculate class imbalance ratio for XGBoost
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print(f"Class imbalance ratio (scale_pos_weight for XGBoost): {scale_pos_weight:.1f}")


In [ ]:
# Model 1: Logistic Regression (baseline)
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
lr_probs = lr.predict_proba(X_test_scaled)[:, 1]
lr_auc = roc_auc_score(y_test, lr_probs)
print(f"Logistic Regression AUC-ROC: {lr_auc:.4f}")


In [ ]:
# Model 2: Decision Tree
dt = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=100,
    class_weight='balanced',
    random_state=42
)
dt.fit(X_train, y_train)
dt_probs = dt.predict_proba(X_test)[:, 1]
dt_auc = roc_auc_score(y_test, dt_probs)
print(f"Decision Tree AUC-ROC: {dt_auc:.4f}")


In [ ]:
# Model 3: XGBoost
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42,
    verbosity=0
)
xgb.fit(X_train, y_train)
xgb_probs = xgb.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_probs)
print(f"XGBoost AUC-ROC: {xgb_auc:.4f}")


---
## 6. Model Evaluation

In credit risk, **AUC-ROC** and **Gini coefficient** are the standard metrics.  
Accuracy is misleading on imbalanced data -- a model that predicts "no default" every time gets 93% accuracy but is useless.

- **AUC-ROC**: Probability that the model ranks a defaulter higher than a non-defaulter  
- **Gini**: 2 * AUC - 1 (industry standard in credit scoring)  
- **KS Statistic**: Maximum separation between default and non-default score distributions


In [ ]:
# ROC Curves for all three models
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

models = {
    'Logistic Regression': (lr_probs, '#4C72B0'),
    'Decision Tree': (dt_probs, '#55A868'),
    'XGBoost': (xgb_probs, '#DD8452')
}

for name, (probs, color) in models.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    gini = 2 * auc - 1
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f}, Gini={gini:.3f})', color=color, lw=2)

axes[0].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves -- All Models')
axes[0].legend(loc='lower right', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Model comparison bar chart
model_names = list(models.keys())
aucs = [roc_auc_score(y_test, probs) for probs, _ in models.values()]
ginis = [2*a-1 for a in aucs]

x = np.arange(len(model_names))
width = 0.35
bars1 = axes[1].bar(x - width/2, aucs, width, label='AUC-ROC', color='#4C72B0', alpha=0.8)
bars2 = axes[1].bar(x + width/2, ginis, width, label='Gini', color='#DD8452', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, rotation=10)
axes[1].set_ylim(0, 1)
axes[1].set_title('Model Comparison: AUC-ROC vs Gini')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
for bar in bars1:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('model_evaluation_roc.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# KS Statistic for XGBoost (best model)
from scipy import stats

default_scores = xgb_probs[y_test == 1]
non_default_scores = xgb_probs[y_test == 0]

ks_stat, ks_pval = stats.ks_2samp(default_scores, non_default_scores)
print(f"KS Statistic (XGBoost): {ks_stat:.4f}")
print(f"  -- A KS > 0.3 is considered good for credit models")
print(f"  -- A KS > 0.4 is considered very good")

# Score distribution plot
plt.figure(figsize=(10, 5))
plt.hist(non_default_scores, bins=50, alpha=0.6, color='#4C72B0',
         label='Non-Default', density=True)
plt.hist(default_scores, bins=50, alpha=0.6, color='#DD8452',
         label='Default', density=True)
plt.xlabel('Predicted Default Probability')
plt.ylabel('Density')
plt.title(f'Score Distribution by Actual Outcome (XGBoost) -- KS={ks_stat:.3f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ks_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Confusion Matrix for XGBoost at 0.5 threshold
xgb_preds = (xgb_probs >= 0.5).astype(int)
cm = confusion_matrix(y_test, xgb_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Default', 'Default'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix -- XGBoost (threshold=0.5)')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, xgb_preds, target_names=['No Default', 'Default']))


---
## 7. Feature Importance

Understanding which features drive the model is as important as the model's performance.  
In a real credit risk team, you'd present this to stakeholders to explain model decisions.


In [ ]:
# XGBoost Feature Importance
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': xgb.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 7))
colors = ['#DD8452' if imp > importance_df['Importance'].quantile(0.75) else '#4C72B0' for imp in importance_df['Importance']]
plt.barh(importance_df['Feature'], importance_df['Importance'], color=colors)
plt.xlabel('Feature Importance (Gain)')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 5 most important features:")
print(importance_df.tail(5)[['Feature', 'Importance']].to_string(index=False))


---
## 8. Risk Tier Segmentation (Scorecard)

This is the business output. We convert raw model probabilities into a 3-tier risk classification  
that a credit team can actually use to make lending decisions.

- **Low Risk**: predicted default probability < 5%
- **Medium Risk**: 5% to 15%
- **High Risk**: > 15%


In [ ]:
# Build risk scorecard on test set
results = pd.DataFrame({
    'actual_default': y_test.values,
    'default_probability': xgb_probs
})

# Assign risk tiers
def assign_risk_tier(prob):
    if prob < 0.05:
        return 'Low Risk'
    elif prob < 0.15:
        return 'Medium Risk'
    else:
        return 'High Risk'

results['risk_tier'] = results['default_probability'].apply(assign_risk_tier)

# Scorecard summary
scorecard = results.groupby('risk_tier').agg(
    borrower_count=('actual_default', 'count'),
    actual_default_rate=('actual_default', 'mean'),
    avg_predicted_prob=('default_probability', 'mean')
).reindex(['Low Risk', 'Medium Risk', 'High Risk'])

scorecard['borrower_pct'] = scorecard['borrower_count'] / scorecard['borrower_count'].sum() * 100
scorecard['actual_default_rate'] = scorecard['actual_default_rate'] * 100
scorecard['avg_predicted_prob'] = scorecard['avg_predicted_prob'] * 100

print("=== CREDIT RISK SCORECARD ===")
print(scorecard.round(2).to_string())


In [ ]:
# Visualise risk tier distribution and default rates
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

tiers = ['Low Risk', 'Medium Risk', 'High Risk']
colors = ['#55A868', '#CCBA72', '#DD8452']

counts = scorecard['borrower_count'].values
axes[0].bar(tiers, counts, color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_title('Borrower Count by Risk Tier')
axes[0].set_ylabel('Number of Borrowers')
for i, v in enumerate(counts):
    pct = scorecard['borrower_pct'].iloc[i]
    axes[0].text(i, v + 50, str(v) + ' (' + str(round(pct, 1)) + '%)', ha='center', fontsize=10, fontweight='bold')

default_rates = scorecard['actual_default_rate'].values
axes[1].bar(tiers, default_rates, color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_title('Actual Default Rate by Risk Tier')
axes[1].set_ylabel('Default Rate (%)')
for i, v in enumerate(default_rates):
    axes[1].text(i, v + 0.3, str(round(v, 1)) + '%', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Risk Tier Scorecard -- XGBoost Model', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('risk_scorecard.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 9. Summary and Key Findings

### Model Performance

| Model | AUC-ROC | Gini |
|-------|---------|------|
| Logistic Regression | ~0.XX | ~0.XX |
| Decision Tree | ~0.XX | ~0.XX |
| **XGBoost** | **~0.XX** | **~0.XX** |

*(Fill in actual values after running)*

### Key Risk Drivers (from Feature Importance)
1. **Revolving utilization** -- borrowers using >80% of available credit are significantly more likely to default
2. **Total delinquencies** -- past late payment behaviour is the strongest predictor of future default
3. **Debt-to-income ratio** -- higher debt burden relative to income increases default risk
4. **Age** -- younger borrowers carry higher risk

### Risk Tier Segmentation
- **Low Risk** borrowers (~X% of portfolio) have a default rate of ~X%
- **Medium Risk** borrowers (~X% of portfolio) have a default rate of ~X%
- **High Risk** borrowers (~X% of portfolio) have a default rate of ~X%

This segmentation can directly inform lending decisions:
- Low Risk: approve at standard rates
- Medium Risk: approve with higher interest rates or lower credit limits
- High Risk: reject or require additional verification

### Business Recommendation
The XGBoost model with a Gini of ~0.XX is production-ready for a first-generation credit scoring model.  
Next steps would include: calibration testing, champion-challenger framework, and monitoring for score drift over time.


---
# Part 2: Fraud Detection

Credit risk and fraud are related but distinct problems.
- **Credit risk**: will this borrower fail to repay?
- **Fraud**: is this borrower misrepresenting themselves to obtain credit?

We use two approaches:
1. **Rule-based flagging**: encodes domain knowledge about known fraud patterns
2. **Isolation Forest**: flags statistically anomalous profiles without labels


---
## 10. Fraud Signal Analysis

Common fraud signals in credit applications:
- High income + high utilization (income inflation)
- Very young borrower with many open credit lines (identity theft)
- Many real estate loans + high delinquency (loan stacking)
- Suspiciously round income numbers


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)

fraud_df = df.copy()
print('Working dataset:', fraud_df.shape)


In [ ]:
# Create fraud-specific features
fraud_df['income_utilization_flag'] = (
    (fraud_df['monthly_income'] > fraud_df['monthly_income'].quantile(0.9)) &
    (fraud_df['revolving_utilization'] > 0.7)
).astype(int)

fraud_df['identity_theft_signal'] = (
    (fraud_df['age'] < 22) & (fraud_df['open_credit_lines'] > 5)
).astype(int)

fraud_df['stacking_signal'] = (
    (fraud_df['real_estate_loans'] > 3) & (fraud_df['total_delinquencies'] > 2)
).astype(int)

fraud_df['round_income_flag'] = (fraud_df['monthly_income'] % 1000 == 0).astype(int)

fraud_df['rule_fraud_score'] = (
    fraud_df['income_utilization_flag'] +
    fraud_df['identity_theft_signal'] +
    fraud_df['stacking_signal'] +
    fraud_df['round_income_flag']
)

fraud_df['rule_fraud_flag'] = (fraud_df['rule_fraud_score'] >= 2).astype(int)

signals = ['income_utilization_flag','identity_theft_signal','stacking_signal','round_income_flag','rule_fraud_flag']
for s in signals:
    c = fraud_df[s].sum()
    print(f'{s}: {c:,} ({c/len(fraud_df)*100:.2f}%)')


In [ ]:
# Default rate by each fraud signal
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
signal_labels = [
    ('income_utilization_flag', 'High Income + High Utilization'),
    ('identity_theft_signal', 'Young Borrower + Many Credit Lines'),
    ('stacking_signal', 'Many Real Estate Loans + Delinquency'),
    ('round_income_flag', 'Suspiciously Round Income'),
]
for ax, (col, label) in zip(axes.flatten(), signal_labels):
    rates = fraud_df.groupby(col)['default'].mean() * 100
    bars = ax.bar(['Not Flagged', 'Flagged'], rates.values, color=['#4C72B0','#DD8452'], edgecolor='white')
    ax.set_title(label, fontsize=11)
    ax.set_ylabel('Default Rate (%)')
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'{bar.get_height():.1f}%', ha='center', fontweight='bold')
plt.suptitle('Default Rate by Fraud Signal', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('fraud_signals.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 11. Isolation Forest -- Anomaly Detection

Isolation Forest detects anomalies by randomly partitioning data.
Anomalous records are isolated in fewer steps because they are statistically unusual.

**Advantage:** Works without labeled fraud data.
**Limitation:** Anomalous does not always mean fraudulent.


In [ ]:
anomaly_features = [
    'revolving_utilization','age','monthly_income','debt_ratio',
    'open_credit_lines','real_estate_loans','dependents',
    'total_delinquencies','dti_ratio','high_utilization'
]

X_fraud = fraud_df[anomaly_features].copy()
scaler_fraud = StandardScaler()
X_fraud_scaled = scaler_fraud.fit_transform(X_fraud)

iso_forest = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1)
iso_forest.fit(X_fraud_scaled)

fraud_df['anomaly_score'] = iso_forest.decision_function(X_fraud_scaled)
fraud_df['iso_fraud_flag'] = (iso_forest.predict(X_fraud_scaled) == -1).astype(int)

flagged = fraud_df['iso_fraud_flag'].sum()
print(f'Flagged: {flagged:,} ({flagged/len(fraud_df)*100:.1f}%)')
print(f'Default rate -- flagged: {fraud_df[fraud_df["iso_fraud_flag"]==1]["default"].mean():.2%}')
print(f'Default rate -- normal:  {fraud_df[fraud_df["iso_fraud_flag"]==0]["default"].mean():.2%}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(fraud_df[fraud_df['iso_fraud_flag']==0]['anomaly_score'], bins=60, alpha=0.6, color='#4C72B0', label='Normal', density=True)
axes[0].hist(fraud_df[fraud_df['iso_fraud_flag']==1]['anomaly_score'], bins=60, alpha=0.6, color='#DD8452', label='Anomalous', density=True)
axes[0].axvline(x=0, color='black', linestyle='--', lw=1.5, label='Decision boundary')
axes[0].set_xlabel('Anomaly Score')
axes[0].set_title('Isolation Forest Score Distribution')
axes[0].legend()

groups = ['Normal', 'Anomalous']
rates = [fraud_df[fraud_df['iso_fraud_flag']==0]['default'].mean()*100,
         fraud_df[fraud_df['iso_fraud_flag']==1]['default'].mean()*100]
bars = axes[1].bar(groups, rates, color=['#4C72B0','#DD8452'], edgecolor='white')
axes[1].set_title('Default Rate: Normal vs Anomalous')
axes[1].set_ylabel('Default Rate (%)')
for bar in bars:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f'{bar.get_height():.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('isolation_forest.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Profile anomalous vs normal borrowers
profile_cols = ['revolving_utilization','monthly_income','age','debt_ratio','total_delinquencies','open_credit_lines']
comparison = pd.DataFrame({
    'Normal': fraud_df[fraud_df['iso_fraud_flag']==0][profile_cols].mean(),
    'Anomalous': fraud_df[fraud_df['iso_fraud_flag']==1][profile_cols].mean()
}).round(2)
comparison['Diff %'] = ((comparison['Anomalous'] - comparison['Normal']) / comparison['Normal'] * 100).round(1).astype(str) + '%'
print('=== ANOMALOUS BORROWER PROFILE ===\n')
print(comparison.to_string())


---
## 12. Combined Credit Risk + Fraud Decision Matrix

In a real credit team, both scores are used together.
A borrower can be low credit risk but high fraud risk (stolen identity with good credit history).
The combined view catches what either model alone would miss.


In [ ]:
# Generate predictions for full dataset for combined scorecard
fraud_df['credit_risk_prob'] = xgb.predict_proba(fraud_df[feature_cols])[:, 1]

def credit_tier(prob):
    if prob < 0.05: return 'Low Credit Risk'
    elif prob < 0.15: return 'Medium Credit Risk'
    else: return 'High Credit Risk'

fraud_df['credit_tier'] = fraud_df['credit_risk_prob'].apply(credit_tier)

def combined_decision(row):
    if row['iso_fraud_flag'] == 1 or row['rule_fraud_flag'] == 1:
        return 'Review -- Fraud Flag'
    elif row['credit_tier'] == 'High Credit Risk':
        return 'Reject'
    elif row['credit_tier'] == 'Medium Credit Risk':
        return 'Approve with Conditions'
    else:
        return 'Approve'

fraud_df['lending_decision'] = fraud_df.apply(combined_decision, axis=1)

summary = fraud_df.groupby('lending_decision').agg(
    count=('default','count'),
    default_rate=('default','mean')
).sort_values('count', ascending=False)
summary['pct'] = (summary['count']/len(fraud_df)*100).round(1)
summary['default_rate'] = (summary['default_rate']*100).round(2)
print('=== COMBINED LENDING DECISION MATRIX ===')
print(summary.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
decisions = summary.index.tolist()
color_map = {'Approve':'#55A868','Approve with Conditions':'#CCBA72','Reject':'#DD8452','Review -- Fraud Flag':'#C44E52'}
colors = [color_map.get(d,'#4C72B0') for d in decisions]

axes[0].bar(decisions, summary['pct'].values, color=colors, edgecolor='white')
axes[0].set_title('Portfolio Split by Lending Decision')
axes[0].set_ylabel('% of Portfolio')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(summary['pct'].values):
    axes[0].text(i, v + 0.3, str(round(v, 1)) + '%', ha='center', fontweight='bold')

axes[1].bar(decisions, summary['default_rate'].values, color=colors, edgecolor='white')
axes[1].set_title('Actual Default Rate by Decision')
axes[1].set_ylabel('Default Rate (%)')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(summary['default_rate'].values):
    axes[1].text(i, v + 0.2, str(round(v, 1)) + '%', ha='center', fontweight='bold')

plt.suptitle('Combined Credit Risk + Fraud Detection Decision Matrix', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('combined_decision.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 13. Final Project Summary

### What this project covers

| Component | Technique | Business Purpose |
|-----------|-----------|------------------|
| Credit default prediction | Logistic Regression, Decision Tree, XGBoost | Predict which borrowers will default |
| Risk tier segmentation | Probability thresholds | Translate model output into lending decisions |
| Fraud signal detection | Rule-based flagging | Flag known fraud patterns |
| Anomaly detection | Isolation Forest | Catch unusual profiles without labels |
| Combined decision matrix | Credit + fraud scores | Final approve/reject/review per borrower |

### Model Performance (fill in after running)

| Model | AUC-ROC | Gini | KS Stat |
|-------|---------|------|---------|
| Logistic Regression | | | |
| Decision Tree | | | |
| XGBoost | | | |

### Key Takeaways
1. Past delinquency behaviour is the strongest predictor of default
2. XGBoost outperforms simpler models significantly on credit data
3. Fraud detection needs both rules (known patterns) and models (unknown anomalies)
4. The combined scorecard reduces both credit losses and fraud losses in one framework

### Production next steps
- Calibration testing
- Champion-challenger framework against existing model
- Monthly score distribution monitoring for drift
- Reject inference to handle selection bias in approved-only data
